<a href="https://colab.research.google.com/github/Makena-mel/Makena-mel/blob/main/Fruits_Molten_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**The convolutional Layer**


In [10]:
!pip install tensorflow

In [12]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(3,3), activation='relu', input_shape=(100,100,3)))
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 98, 98, 32)     │           896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 896 (3.50 KB)

 Trainable params: 896 (3.50 KB)

 Non-trainable params: 0 (0.00 B)

This is the foundational building block of CNNs and is well-suited for image classification like the fruit recognition task. It works by sliding kernels across an input image which captures spatial patterns like edges, shapes and textures.In the context of fruit classificatuion, this is done by assessing the different visual cues like shape outlines, patterns, speckles or the surface textures and colour gradients, which are different from one fruit to another. Early convolutional layers learn the low-level features(like color transitions ) while deeper layers combine them into higher-level classifications which are more discriminitive for different fruit categories. The Fruits-360 dataset contains 260 classes of images with 100x100 pixels.Because the background is controlled( a white sheet of paper), the convolutional layer focuses its learning capacity on the fruit. The local connectivity and weight sharing of convolutional layers mean the same filter is applied across the entire image, aproperty known as transition invariance which is critical since the fruit image positions may vary.Convolutional layers drastically reduce the number of parameters as compared to FC layers, making the model less prone to overfitting on a dataset.
In the case of fruits-360 dataset, ANNs struggle because they convert the input into a 1-dimension array.The convolutional layer solves this by taking into account the 2-dimensional structure, hence the rotating of the fruits.The kernels in the convolutional layer then slide across the width and height of the fruit's image to extract high-level features which are used to produce an activation map.This helps the network learn specific features of the fruit and precise location on the 100x100 pixels frame.

In [14]:

#Simulated batch of fruit data: image input with 100x100 pixels with 3 channels
fruit_batch = tf.random.uniform(shape=(10,100,100,3), minval=0, maxval=1)
#Convolutional layer with 32 filters, kernel size 3x3 same size padding as the data
convo_layer = tf.keras.layers.Conv2D(filters=32, kernel_size= (3,3), padding='same', activation='relu')
#pass the fruit batch through the convolutional layer
convo_out = convo_layer(fruit_batch)
print("Original image shape(Batch, Height, Width, Channels):",fruit_batch.shape)
print("Convolved image shape(Batch, Height, Width, Channels):",convo_out.shape)


Original image shape(Batch, Height, Width, Channels): (10, 100, 100, 3)
Convolved image shape(Batch, Height, Width, Channels): (10, 100, 100, 32)


**The** **Pooling** **Layer**

In [15]:
from tensorflow.keras.layers import MaxPooling2D
model.add(MaxPooling2D(pool_size=(2,2)))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 896 (3.50 KB)

 Trainable params: 896 (3.50 KB)

 Non-trainable params: 0 (0.00 B)

This are downsampling operations inserted after the convolutional layers.The most common form is Maxpooling which divides the feature map into non-overlapping rectangular regions and outputs only the maximum value from each, unlike the average pooling which outputs the mean.
In this context(fruits classification), pooling layers serve critical functions like,to reduce the spatial dimensions of the data representation which then reduces the huge amount of computation costs required by the network, which is important when training such a large dataset(Fruits-360 dataset).Pooling also introduces a degree of translation invariance, which is important in classification and accuracy.As the fruit is being rotated, a distinctive texture may appear in one image and not in one.. In such a case, pooling operations ensure that the presence of the texture is captured regardless of its exact location within the pooled region. Pooling also reduces overfitting by abstrating the feature maps, instead of memorising the exact pixel-level patterns. This is useful in the fruits-360 dataset since each class has multiple images taken throughout the different rotations.By reducing the spatial resolution, pooling layers allow deeper layers of the network to operate on abstract representions of the fruit, which is helpful when distinguishing visually similar fruits.

In [20]:
# Define a Max Pooling Layer: 3x2 pool size with 4 strides
pool_layer= tf.keras.layers.MaxPooling2D(pool_size=(3,2),strides =4)
#Pass the 32 feature maps
pool_features= pool_layer(convo_out)
print("feature map shape before pooling:",convo_out.shape)
print("feature map shape after pooling:",pool_features.shape)
#

feature map shape before pooling: (10, 100, 100, 32)
feature map shape after pooling: (10, 25, 25, 32)


**Loss Layer**

In [21]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

This layer quantifies how far the model's predictionsare from the true labels during training. It produces a scalar value which is then used by the optimiser(adam or stochastic gradient descent) to compute gradients and update the networks weights via backpropagation.Categorical cross-entropy is the standard for a multi-class classification such as the Fruits-360 dataset.This function compares the model's predicted probability distribution against the encoded ground true label. In fruits classification, the loss layer plays a pivotal role in shaping what the model learns.Poorly choosen or exexcuted loss layer may cause the network to fail to distinguish between visually similar categories. The categorical cross-entropy ensures that the model is penalised proportionally to its confidence in incorrect class predictions, encouraging sharper discrimination between classes.It also interacts with class imbalance.If some fruit classes have fewer training images, the loss can be weighted to penalise misclassifications of underrepresented classes hence improving fairness across all classes.Monitoring the loss value over epochs during training is essential for diagnosing overfitting, underfitting and convergence.

In [24]:
#simulated the final classification layer outputing raw scores for 131 fruit classes
lo_logits= tf.random.normal(shape=(1,131))
true_label=tf.constant([42])
loss_function= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
#calculate the error penalty
error_penalty= loss_function(true_label,lo_logits)
print("Simulated logits shape(1 image, 131 classes):", lo_logits.shape)
print("Error penalty(loss):",error_penalty.numpy())


Simulated logits shape(1 image, 131 classes): (1, 131)
Error penalty(loss): 3.98534


**Deep Belief Network**

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
model =Sequential ([
    Flatten(input_shape=(100,100,3)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(131, activation='softmax')

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


It is a generative probabilistic model composed of multiple layers of stochastic variables.It is built by stacking RBMs where each of them are trained in an unsupervised layer-by-layer fashion to learn a compressed representation of its input. This process is important as it cn fundamentally improve the initial values of the network's weights before formally beginning supervised training, which then improves the overall quality of the network and reduce the time taken for training.It can be fine-tuned using a supervised signal via backpropagation.For fruit classification, DBN offer several advantages especially in scenarios where labelled data is limited.The unsupervised pre-training phase allows for the model to learn meaningful internal representations of fruit images without requiring labels.The pre-training also acts as a weight initialization strategy that places the network's parameters in a region of the loss landscape condusive to good generalization.They can also function as feature extractors since they learn the hidden representions and feed them to a classifier(softmax layer) for the final fruit category prediction.

In [28]:
import numpy as np
#Neural network initialization
#standard networks assign completely random noise to their starting weights
stand_layer=tf.keras.layers.Dense(256, activation='relu')
stand_layer.build((None, 1024))

#Simulating DBN pre-training phase
#generate a mathematically scaled array of optimized weights
dbn_weights=[
    np.random.normal(loc=0, scale=0.05, size=(1024,256)).astype(np.float32),#optimize weights
    np.zeros(256).astype(np.float32)#basics
]
#create a new layer but now with the dbn's pre-trained weights
dbn_pretrained=tf.keras.layers.Dense(256, activation='relu')
dbn_pretrained.build((None, 1024))
dbn_pretrained.set_weights(dbn_weights)
print('Standard layr weights(first 4):', stand_layer.get_weights()[0][0][:4])
print("DBN pre-trained layer weights (first 8):", dbn_pretrained.get_weights()[0][0][:4])
print("\n The DBN mathematically organizes the weights before training even begins, preventing the network from guessing blindly")

Standard layr weights(first 4): [ 0.02967878 -0.03551134  0.00911471  0.03769691]
DBN pre-trained layer weights (first 8): [-0.03359875 -0.00184482  0.05405442 -0.02173277]

 The DBN mathematically organizes the weights before training even begins, preventing the network from guessing blindly
